In [6]:
import pandas as pd
import re

# 1. Đọc dữ liệu
df = pd.read_csv('/kaggle/input/datasets/labrynthlabrynth/dataset/data_cleaned_final.csv')

# 2. Định nghĩa patterns
date_patterns = [
    r'\b\d{1,2}\s+tháng\s+\d{1,2}\s+năm\s+\d{4}\b',
    r'\b\d{1,2}\s+tháng\s+\d{1,2}\b',
    r'\b\d{1,2}[\.\/\-]\d{1,2}[\.\/\-]\d{2,4}\b',
    r'\bnăm\s+\d{4}\b',
    r'(?<!\d)\b(1[0-9]|20)\d{2}\b(?!\s*(người|km|kg|m|KB|MB|GB))'
]
combined_pattern = '|'.join(date_patterns)

def mask_time(text):
    if pd.isna(text):
        return text
    return re.sub(combined_pattern, '[MASK]', str(text), flags=re.IGNORECASE)

# 3. Sort theo Năm trước khi đánh thứ tự (fix lỗi đã nhận xét trước)
df = df.sort_values(by=['Nhân vật', 'Năm']).reset_index(drop=True)
df['Thứ tự'] = df.groupby('Nhân vật').cumcount() + 1

# 4. Chọn ngẫu nhiên 1 index của mỗi nhân vật để mask
random_masked_idx = (
    df.groupby('Nhân vật')
      .apply(lambda g: g.sample(n=1, random_state=42).index[0])
)
# random_masked_idx là Series: {tên_nhân_vật: index_trong_df}
masked_indices = set(random_masked_idx.values)

# 5. Tạo cột Sự kiện_Masked:
#    - Đúng hàng được chọn → apply mask_time
#    - Các hàng còn lại     → giữ nguyên Sự kiện gốc
df['Sự kiện_Masked'] = df.apply(
    lambda row: mask_time(row['Sự kiện']) if row.name in masked_indices else row['Sự kiện'],
    axis=1
)

# 6. Thêm cột đánh dấu để dễ kiểm tra
df['Is_Masked'] = df.index.isin(masked_indices)

# 7. Xuất
cols = ['Nhân vật', 'Thứ tự', 'Năm', 'Sự kiện', 'Sự kiện_Masked', 'Is_Masked']
df_out = df[cols] if all(c in df.columns for c in cols) else df
df_out.to_csv('data_processed.csv', index=False)

# 8. Kiểm tra nhanh
print("Đã xử lý xong!")
print(f"\nTổng số nhân vật: {df['Nhân vật'].nunique()}")
print(f"Số hàng được mask:  {df['Is_Masked'].sum()} (= 1 sự kiện/người ✓)")
print("\n--- Ví dụ 3 nhân vật đầu ---")
sample = df[df['Is_Masked']].head(3)[['Nhân vật', 'Thứ tự', 'Sự kiện', 'Sự kiện_Masked']]
print(sample.to_string(index=False))

KeyboardInterrupt: 

In [ ]:
# CELL 1: CÀI ĐẶT VÀ IMPORT THƯ VIỆN (BẢN AN TOÀN CHO KAGGLE)

# Chỉ cài thêm transformers nếu thiếu, không phá vỡ môi trường Kaggle gốc
!pip install -q transformers tqdm

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Bỏ AdamW ra khỏi transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Import AdamW từ thư viện gốc của PyTorch
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import random

# Cố định random seed để kết quả ổn định qua nhiều lần chạy
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")
df = pd.read_csv('/kaggle/working/data_processed.csv')
df = df.sort_values(by=['Nhân vật', 'Thứ tự'])

# 1. Tách 80% Nhân vật cho Train, 20% cho Test
all_characters = df['Nhân vật'].unique()
train_chars, test_chars = train_test_split(all_characters, test_size=0.2, random_state=42)

df_train = df[df['Nhân vật'].isin(train_chars)]
df_test = df[df['Nhân vật'].isin(test_chars)]

print(f"Tổng số nhân vật: {len(all_characters)} | Train: {len(train_chars)} | Test: {len(test_chars)}")

# 2. Tạo dữ liệu Train Pairwise (Chỉ dùng tập df_train, kết hợp Window Sampling)
pairs = []
labels = []
WINDOW_SIZE = 4

for name, group in tqdm(df_train.groupby('Nhân vật'), desc="Tạo Train Data"):
    events = group['Sự kiện_Masked'].tolist()
    n = len(events)
    for i in range(n):
        end_idx = min(i + 1 + WINDOW_SIZE, n)
        for j in range(i + 1, end_idx):
            pairs.append((f"{name} - {events[i]}", events[j]))
            labels.append(1)
            pairs.append((f"{name} - {events[j]}", events[i]))
            labels.append(0)

train_df = pd.DataFrame({'text_a': [p[0] for p in pairs], 'text_b': [p[1] for p in pairs], 'label': labels})
print(f"Tạo được {len(train_df)} cặp câu cho tập Train.")

In [ ]:
# =======================================================
# CELL: TÍCH HỢP TRAIN + VALIDATION + XUẤT 6 BIỂU ĐỒ & FILE
# =======================================================

import warnings
import torch
import torch.nn as nn
import scipy.stats as stats
import numpy as np
import pandas as pd
import random
import gc
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from transformers.utils import logging
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore")
logging.set_verbosity_error()

plt.style.use('seaborn-v0_8-whitegrid')
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

# =======================================================
#  CẤU HÌNH HỆ THỐNG
# =======================================================
# =======================================================
#  CẤU HÌNH HỆ THỐNG — KAGGLE PATHS
# =======================================================
FAST_MODE = False
EVENT_THRESHOLD = 10

# --- INPUT paths (read-only trên Kaggle) ---
INPUT_DIR         = "/kaggle/input/datasets/labrynthlabrynth/checkpoint"   # ← đổi tên dataset của bạn
INPUT_BEST_MODEL  = '/kaggle/input/models/labrynthlabrynth/phobert/other/default/1'
INPUT_CHECKPOINT  = os.path.join(INPUT_DIR, "last_checkpoint.pth")
INPUT_METRICS     = os.path.join(INPUT_DIR, "evaluation_metrics.csv")

# --- OUTPUT paths (writable) ---
WORK_DIR              = "/kaggle/working"
SAVE_PATH             = os.path.join(WORK_DIR, "best_model")
CHECKPOINT_FILE       = os.path.join(WORK_DIR, "last_checkpoint.pth")
EVAL_METRICS_FILE     = os.path.join(WORK_DIR, "evaluation_metrics.csv")
EVAL_PREDICTIONS_FILE = os.path.join(WORK_DIR, "evaluation_predictions.csv")

os.makedirs(SAVE_PATH, exist_ok=True)

positive_train_df = train_df[train_df['label'] == 1].reset_index(drop=True)
if FAST_MODE:
    print(" ĐANG CHẠY CHẾ ĐỘ NHÁP")
    train_subset = positive_train_df.sample(n=10000, random_state=42)
    VAL_MAX_ROWS = 2000
else:
    print(" ĐANG CHẠY CHẾ ĐỘ FULL")
    train_subset = positive_train_df
    VAL_MAX_ROWS = None


# =======================================================
#  HÀM BUILD PROMPT TRUNG LẬP (NLI FRAMING)
# Dùng chung cho cả train lẫn inference — không leak nhãn
# =======================================================
def build_prompt(char_name, event_a, event_b):
    """
    Trả về (premise, hypothesis) theo format NLI trung lập.
    event_a là sự kiện được giả định xảy ra trước event_b.
    Hypothesis giữ nguyên — model buộc phải dựa vào nội dung premise.
    """
    premise = (
        f"Tiểu sử {char_name}. "
        f"Sự kiện X: {event_a} | Sự kiện Y: {event_b}"
    )
    hypothesis = "Sự kiện X xảy ra trước Sự kiện Y."
    return premise, hypothesis


# =======================================================
# 1. DATASET & DATALOADER
# =======================================================
class RankingEventDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        parts = str(row['text_a']).split(" - ", 1)
        char_name = parts[0] if len(parts) == 2 else "Nhân vật"
        event_a   = parts[1] if len(parts) == 2 else str(row['text_a'])
        event_b   = str(row['text_b'])

        #  Dùng build_prompt — không còn leak nhãn
        pos_text_a, pos_text_b = build_prompt(char_name, event_a, event_b)
        neg_text_a, neg_text_b = build_prompt(char_name, event_b, event_a)

        pos_enc = self.tokenizer(
            pos_text_a, pos_text_b,
            truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        neg_enc = self.tokenizer(
            neg_text_a, neg_text_b,
            truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )

        return {
            'pos_input_ids':      pos_enc['input_ids'].flatten(),
            'pos_attention_mask': pos_enc['attention_mask'].flatten(),
            'neg_input_ids':      neg_enc['input_ids'].flatten(),
            'neg_attention_mask': neg_enc['attention_mask'].flatten(),
        }



# =======================================================
# 2. HÀM ĐÁNH GIÁ
# =======================================================
def evaluate_timeline_batched(character_name, original_events, model, tokenizer, device, batch_size=128):
    n = len(original_events)
    if n < 2:
        return 1.0, 1.0, list(range(n)), list(range(n)), [], []

    shuffled_indices = list(range(n))
    rng = np.random.default_rng(seed=42)
    rng.shuffle(shuffled_indices)
    shuffled_events = [original_events[i] for i in shuffled_indices]

    pairs, pair_indices = [], []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            #  Dùng build_prompt — nhất quán với train
            text_a, text_b = build_prompt(character_name, shuffled_events[i], shuffled_events[j])
            pairs.append((text_a, text_b))
            pair_indices.append((i, j))

    borda_scores = np.zeros(n)
    for k in range(0, len(pairs), batch_size):
        batch_pairs = pairs[k:k + batch_size]
        inputs = tokenizer(
            [p[0] for p in batch_pairs],
            [p[1] for p in batch_pairs],
            return_tensors='pt', truncation=True,
            max_length=128, padding='max_length'
        )
        with torch.no_grad():
            scores = model(
                input_ids=inputs['input_ids'].to(device),
                attention_mask=inputs['attention_mask'].to(device)
            ).logits.squeeze(-1).cpu().numpy()

        for idx, score in enumerate(scores):
            i, j = pair_indices[k + idx]
            borda_scores[i] += score

        del inputs, scores

    torch.cuda.empty_cache()

    pred_order_indices = np.argsort(borda_scores)[::-1]
    pred_mapped  = [shuffled_indices[idx] for idx in pred_order_indices]
    ground_truth = list(range(n))

    pred_ranks = np.zeros(n, dtype=int)
    for rank, original_idx in enumerate(pred_mapped):
        pred_ranks[original_idx] = rank + 1
    true_ranks = np.arange(1, n + 1)

    tau, _ = stats.kendalltau(ground_truth, pred_mapped)
    if np.isnan(tau):
        tau = sum(1 for i, j in zip(ground_truth, pred_mapped) if i == j) / n

    rho, _ = stats.spearmanr(true_ranks, pred_ranks)
    if np.isnan(rho):
        rho = 1.0

    return tau, rho, ground_truth, pred_mapped, true_ranks.tolist(), pred_ranks.tolist()


def validate_model(current_model, test_df, tokenizer, device, max_rows, epoch_num=None):
    eval_df = test_df
    if max_rows is not None:
        selected_chars, row_count = [], 0
        for char in test_df['Nhân vật'].unique():
            selected_chars.append(char)
            row_count += len(test_df[test_df['Nhân vật'] == char])
            if row_count >= max_rows:
                break
        eval_df = test_df[test_df['Nhân vật'].isin(selected_chars)]

    scores, spearman_scores, short_scores, long_scores = [], [], [], []
    exact_matches = 0
    all_true_ranks, all_pred_ranks, detailed_results = [], [], []
    sample_timeline = None
    current_model.eval()

    val_loop = tqdm(eval_df.groupby('Nhân vật'), desc=f" Đang Test (Epoch {epoch_num})", leave=False)
    for name, group in val_loop:
        events     = group['Sự kiện_Masked'].tolist()
        num_events = len(events)
        if num_events < 3:
            continue

        tau, rho, truth, pred, t_ranks, p_ranks = evaluate_timeline_batched(
            name, events, current_model, tokenizer, device
        )

        scores.append(tau)
        spearman_scores.append(rho)
        if truth == pred:
            exact_matches += 1
        if num_events <= EVENT_THRESHOLD:
            short_scores.append(tau)
        else:
            long_scores.append(tau)

        all_true_ranks.extend(t_ranks)
        all_pred_ranks.extend(p_ranks)

        for i in range(num_events):
            detailed_results.append({
                "Nhân vật": name,
                "Sự kiện":  events[i],
                "Thực tế":  t_ranks[i],
                "Dự đoán":  p_ranks[i],
                "Kết quả":  "" if t_ranks[i] == p_ranks[i] else ""
            })

        if sample_timeline is None and 6 <= num_events <= 10:
            sample_timeline = (name, events, t_ranks, p_ranks)

        val_loop.set_postfix(Tau=f"{np.mean(scores):.4f}", Rho=f"{np.mean(spearman_scores):.4f}")

    return (
        np.mean(scores),
        np.mean(spearman_scores),
        (exact_matches / len(scores)) * 100 if scores else 0,
        np.mean(short_scores) if short_scores else 0.0,
        np.mean(long_scores)  if long_scores  else 0.0,
        all_true_ranks, all_pred_ranks,
        detailed_results, sample_timeline
    )


# =======================================================
# 3. VÒNG LẶP HUẤN LUYỆN CHÍNH
# =======================================================
EPOCHS, PATIENCE = 20, 3

BASE_PRETRAINED = "vinai/phobert-base-v2"

if os.path.exists(INPUT_BEST_MODEL) and os.path.isdir(INPUT_BEST_MODEL):
    print(f" Load model từ input best_model: {INPUT_BEST_MODEL}")
    model = AutoModelForSequenceClassification.from_pretrained(
        INPUT_BEST_MODEL, num_labels=1
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(INPUT_BEST_MODEL)
else:
    print(f" Không tìm thấy best_model input → Load pretrained: {BASE_PRETRAINED}")
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_PRETRAINED, num_labels=1
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(BASE_PRETRAINED)

for module in model.roberta.encoder.layer[:6]:
    for param in module.parameters():
        param.requires_grad = False
        
tokenizer   = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = DataLoader(
    RankingEventDataset(train_subset, tokenizer),
    batch_size=16, shuffle=True, pin_memory=True, num_workers=2
)
optimizer      = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=3e-5
)
margin_loss_fn = nn.MarginRankingLoss(margin=1.0)

#  Khai báo scheduler SAU optimizer, dùng tên nhất quán "scheduler"
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',     # Kendall Tau — cao hơn tốt hơn
    factor=0.5,     # giảm LR xuống 50% khi không cải thiện
    patience=2,
    min_lr=1e-6
)

# =======================================================
#  KHÔI PHỤC CHECKPOINT & METRICS TỪ INPUT
# =======================================================

# Copy metrics từ input → working để append tiếp
if os.path.exists(INPUT_METRICS) and not os.path.exists(EVAL_METRICS_FILE):
    import shutil
    shutil.copy2(INPUT_METRICS, EVAL_METRICS_FILE)
    print(f" Đã copy metrics từ input: {INPUT_METRICS}")

history_logs = []
if os.path.exists(EVAL_METRICS_FILE):
    history_logs = pd.read_csv(EVAL_METRICS_FILE).to_dict('records')
    print(f" Loaded {len(history_logs)} epoch logs từ metrics file.")

# Xác định nguồn checkpoint: ưu tiên working (nếu đang chạy dở),
# fallback về input (lần chạy mới hoàn toàn)
RESOLVED_CHECKPOINT = None
if os.path.exists(CHECKPOINT_FILE):
    RESOLVED_CHECKPOINT = CHECKPOINT_FILE
    print(f" Dùng checkpoint từ working: {CHECKPOINT_FILE}")
elif os.path.exists(INPUT_CHECKPOINT):
    RESOLVED_CHECKPOINT = INPUT_CHECKPOINT
    print(f" Dùng checkpoint từ input: {INPUT_CHECKPOINT}")

start_epoch, best_val_tau, patience_counter = 0, -1.0, 0
best_true_ranks, best_pred_ranks, best_detailed_results, best_sample_timeline = [], [], [], None

if RESOLVED_CHECKPOINT:
    print(f" Đang khôi phục từ: {RESOLVED_CHECKPOINT}")
    checkpoint = torch.load(RESOLVED_CHECKPOINT, map_location=device, weights_only=False)
    
    # Chỉ load optimizer/scheduler state nếu model load từ checkpoint cùng nguồn
    # (tránh mismatch nếu model load từ best_model còn checkpoint từ nơi khác)
    try:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print(" Đã restore optimizer + scheduler state.")
    except Exception as e:
        print(f" Bỏ qua optimizer/scheduler state ({e}) → dùng mặc định.")

    start_epoch      = checkpoint['epoch'] + 1
    best_val_tau     = checkpoint['best_val_tau']
    patience_counter = checkpoint['patience_counter']

    print(f" Resume từ epoch {start_epoch} | Best Tau: {best_val_tau:.4f} | Patience: {patience_counter}/{PATIENCE}")
else:
    print(" Không có checkpoint")

print(" BẮT ĐẦU HUẤN LUYỆN VÀ ĐÁNH GIÁ ĐỒNG THỜI...")
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_train_loss = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", mininterval=10.0)

    for batch in train_loop:
        optimizer.zero_grad()
        pos_scores = model(
            input_ids=batch['pos_input_ids'].to(device),
            attention_mask=batch['pos_attention_mask'].to(device)
        ).logits
        neg_scores = model(
            input_ids=batch['neg_input_ids'].to(device),
            attention_mask=batch['neg_attention_mask'].to(device)
        ).logits

        loss = margin_loss_fn(pos_scores, neg_scores, torch.ones(pos_scores.size(), device=device))
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    val_tau, val_rho, exact_rate, short_tau, long_tau, t_ranks, p_ranks, detailed_results, sample_timeline = \
        validate_model(model, df_test, tokenizer, device, VAL_MAX_ROWS, epoch + 1)

    short_acc  = ((short_tau + 1) / 2) * 100
    long_acc   = ((long_tau  + 1) / 2) * 100
    perf_ratio = (long_acc / short_acc) if short_acc > 0 else 0

    print(f"\n EPOCH {epoch+1} KẾT THÚC | Train Loss: {avg_train_loss:.4f}")
    print(f"   ➤ Kendall Tau: {val_tau:.4f} | Spearman Rho: {val_rho:.4f} | Exact Match: {exact_rate:.2f}%")
    print(f"   ➤ Short Acc: {short_acc:.2f}% | Long Acc: {long_acc:.2f}% | Ratio: {perf_ratio:.2f}")

    log_entry = {
        "Epoch": epoch + 1, "Train_Loss": avg_train_loss,
        "Val_Tau": val_tau, "Val_Spearman": val_rho,
        "Exact_Match_Pct": exact_rate,
        "Acc_Short_Events": short_acc, "Acc_Long_Events": long_acc,
        "Ratio_Long_vs_Short": perf_ratio
    }
    history_logs.append(log_entry)
    pd.DataFrame(history_logs).to_csv(EVAL_METRICS_FILE, index=False)

    if val_tau > best_val_tau:
        best_val_tau, patience_counter = val_tau, 0
        best_true_ranks, best_pred_ranks         = t_ranks, p_ranks
        best_detailed_results, best_sample_timeline = detailed_results, sample_timeline
        model.save_pretrained(SAVE_PATH)
        tokenizer.save_pretrained(SAVE_PATH)
        print(" KỶ LỤC MỚI! Đã lưu Model và Dữ liệu Biểu đồ.")
    else:
        patience_counter += 1
        print(f" Điểm không tăng. Patience: {patience_counter}/{PATIENCE}")

    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),  #  tên nhất quán
        'best_val_tau':         best_val_tau,
        'patience_counter':     patience_counter
    }, CHECKPOINT_FILE)

    #  Gọi scheduler.step với metric (ReduceLROnPlateau yêu cầu truyền giá trị)
    scheduler.step(val_tau)

    if patience_counter >= PATIENCE:
        print(" Early stopping.")
        break

    gc.collect()
    torch.cuda.empty_cache()


# =======================================================
# 4. TỰ ĐỘNG XUẤT 6 BIỂU ĐỒ VÀ OUTPUT
# =======================================================
print("\n ĐANG TẠO 6 BIỂU ĐỒ PHÂN TÍCH TỪ MÔ HÌNH TỐT NHẤT...")
os.system("zip -q -r best_model.zip best_model/")
pd.DataFrame(best_detailed_results).to_csv(EVAL_PREDICTIONS_FILE, index=False, encoding='utf-8-sig')

hist_df = pd.read_csv(EVAL_METRICS_FILE)

# 1. Learning Curve
plt.figure(figsize=(8, 6))
plt.plot(hist_df['Epoch'], hist_df['Train_Loss'], 'r-o', label='Train Loss', linewidth=2)
plt.ylabel('Loss', color='r'); plt.xlabel('Epoch')
ax2 = plt.gca().twinx()
ax2.plot(hist_df['Epoch'], hist_df['Val_Tau'], 'b-o', label='Val Kendall Tau', linewidth=2)
ax2.set_ylabel('Kendall Tau', color='b')
plt.title('1. Learning Curve (Loss vs Kendall Tau)', fontweight='bold')
plt.savefig('plot_1_learning_curve.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Scatter Plot
plt.figure(figsize=(8, 6))
jitter_t = np.array(best_true_ranks) + np.random.normal(0, 0.15, len(best_true_ranks))
jitter_p = np.array(best_pred_ranks) + np.random.normal(0, 0.15, len(best_pred_ranks))
plt.scatter(jitter_t, jitter_p, alpha=0.3, s=20, color='teal')
plt.plot([1, 15], [1, 15], 'r--', linewidth=2)
plt.xlabel('Thực tế'); plt.ylabel('Dự đoán')
plt.title('2. Scatter Plot (Có nhiễu Jitter)', fontweight='bold')
plt.savefig('plot_2_scatter.png', dpi=300, bbox_inches='tight')
plt.close()

# 3. Error Distribution
plt.figure(figsize=(8, 6))
errors = np.array(best_pred_ranks) - np.array(best_true_ranks)
sns.histplot(errors, bins=range(-10, 12), kde=True, color='purple')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Độ lệch (Pred - True)'); plt.ylabel('Tần suất')
plt.title('3. Error Distribution', fontweight='bold')
plt.savefig('plot_3_error_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

# 4. Rank Correlation Density
plt.figure(figsize=(8, 6))
hb = plt.hexbin(best_true_ranks, best_pred_ranks, gridsize=15, cmap='Blues', mincnt=1)
plt.colorbar(hb, label='Số lượng')
plt.plot([1, 15], [1, 15], 'r--', linewidth=2)
plt.xlabel('Thực tế'); plt.ylabel('Dự đoán')
plt.title('4. Rank Correlation Density', fontweight='bold')
plt.savefig('plot_4_rank_density.png', dpi=300, bbox_inches='tight')
plt.close()

# 5. Confusion Matrix
plt.figure(figsize=(8, 6))
max_rank = min(15, max(best_true_ranks) if best_true_ranks else 15)
cm = pd.crosstab(pd.Series(best_true_ranks, name='Thực Tế'), pd.Series(best_pred_ranks, name='Dự Đoán'))
cm = cm.reindex(index=range(1, max_rank+1), columns=range(1, max_rank+1), fill_value=0)
sns.heatmap(cm, cmap='YlGnBu', annot=False, cbar=True)
plt.title('5. Confusion Matrix', fontweight='bold')
plt.savefig('plot_5_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

# 6. Timeline (Bump Chart)
plt.figure(figsize=(8, 6))
if best_sample_timeline:
    char_name, evts, t_ranks, p_ranks = best_sample_timeline
    for i in range(len(evts)):
        plt.plot([0, 1], [t_ranks[i], p_ranks[i]], 'gray', linestyle='-', alpha=0.5)
        plt.plot(0, t_ranks[i], 'bo', markersize=10)
        plt.plot(1, p_ranks[i], 'ro', markersize=10)
        short_text = evts[i][:40] + "..." if len(evts[i]) > 40 else evts[i]
        plt.text(-0.05, t_ranks[i], short_text, ha='right', va='center', fontsize=9)
    plt.xlim(-0.8, 1.2)
    plt.xticks([0, 1], ['Thực tế', 'Dự đoán'], fontweight='bold')
    plt.gca().invert_yaxis()
    plt.axis('off')
    plt.title(f'6. Timeline: {char_name}', fontweight='bold')
plt.savefig('plot_6_timeline.png', dpi=300, bbox_inches='tight')
plt.close()

# =======================================================
# 5. IN KẾT QUẢ
# =======================================================
print("\n" + "="*50)
print(" 1. Lịch sử điểm số (Metrics):              evaluation_metrics.csv")
print(" 2. Kết quả AI dự đoán thực tế:             evaluation_predictions.csv")
print(" 3. Trọng số Model tốt nhất (ZIP):           best_model.zip")
print(" 4. File Checkpoint (Chạy tiếp nếu cần):    last_checkpoint.pth")
print(" 5. Các biểu đồ phân tích:                  plot_1 đến plot_6 (.png)")
print("="*50)